# STRAT-002 Last Year Backtest (2025)

Testing strategy performance for just the last year.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from numba import njit
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("STRAT-002 Last Year Backtest (2025) 📊")

In [ ]:
# Load data
DATA_DIR = Path("../data/daily")

price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")

df = price.join(mvrv, how='inner').join(sopr, how='inner').join(sopr_sth, how='inner').join(realized_loss, how='inner')
df = df.sort_index()

# Create indicators
df['rl_ma30'] = df['realized_loss'].rolling(30).mean()
df['rl_std30'] = df['realized_loss'].rolling(30).std()
df['rl_zscore'] = (df['realized_loss'] - df['rl_ma30']) / df['rl_std30']

# FILTER TO LAST YEAR ONLY (2025)
df = df[df.index >= '2025-01-01'].dropna()

print(f"Data: {len(df)} rows")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")

In [ ]:
# STRAT-002 Parameters
RL_Z_THRESHOLD = 0.5
MVRV_TRIGGER = 2.0
TRAIL_PCT = 0.25
STOP_LOSS_PCT = 0.20

# Entry signal
entry_condition = (
    (df['sopr'] < 1) & 
    (df['sopr_sth'] < 1) & 
    (df['rl_zscore'] > RL_Z_THRESHOLD)
)

entries = entry_condition & ~entry_condition.shift(1).fillna(False)

print(f"Entry signals in 2025: {entries.sum()}")
print(f"\nEntry dates:")
for date in entries[entries].index:
    print(f"  {date.date()}: Price=${df.loc[date, 'price']:,.0f}, MVRV={df.loc[date, 'mvrv']:.2f}")

In [ ]:
@njit
def custom_exit_logic(price_arr, mvrv_arr, entry_idx, 
                      mvrv_trigger=2.0, trail_pct=0.25, stop_loss_pct=0.20):
    entry_price = price_arr[entry_idx]
    peak_price = entry_price
    trailing_active = False
    
    for j in range(entry_idx + 1, len(price_arr)):
        current_price = price_arr[j]
        current_mvrv = mvrv_arr[j]
        
        if current_price > peak_price:
            peak_price = current_price
        
        pnl = (current_price - entry_price) / entry_price
        
        if not trailing_active and current_mvrv >= mvrv_trigger:
            trailing_active = True
        
        if trailing_active:
            trail_stop = peak_price * (1 - trail_pct)
            if current_price <= trail_stop:
                return j, trail_stop, 0
        
        if not trailing_active and pnl <= -stop_loss_pct:
            return j, entry_price * (1 - stop_loss_pct), 1
    
    return len(price_arr) - 1, price_arr[-1], 3

def run_backtest(df, entries, initial_capital=100000, fees=0.001):
    price_arr = df['price'].values
    mvrv_arr = df['mvrv'].values
    dates = df.index
    
    entry_indices = np.where(entries.values)[0]
    
    trades = []
    i = 0
    
    while i < len(entry_indices):
        entry_idx = entry_indices[i]
        
        exit_idx, exit_price, exit_reason = custom_exit_logic(
            price_arr, mvrv_arr, entry_idx,
            MVRV_TRIGGER, TRAIL_PCT, STOP_LOSS_PCT
        )
        
        entry_price = price_arr[entry_idx]
        
        gross_return = (exit_price / entry_price) - 1
        net_return = gross_return - (2 * fees)
        
        exit_reason_map = {0: 'mvrv_trail', 1: 'stop_loss', 3: 'end_of_data'}
        
        trades.append({
            'entry_date': dates[entry_idx],
            'exit_date': dates[exit_idx],
            'entry_price': entry_price,
            'exit_price': exit_price,
            'gross_return': gross_return,
            'net_return': net_return,
            'days_held': exit_idx - entry_idx,
            'exit_reason': exit_reason_map[exit_reason]
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_idx:
            i += 1
    
    trades_df = pd.DataFrame(trades)
    
    if len(trades_df) > 0:
        equity = [initial_capital]
        for _, trade in trades_df.iterrows():
            equity.append(equity[-1] * (1 + trade['net_return']))
        trades_df['equity_after'] = equity[1:]
    
    return trades_df

In [ ]:
# Run backtest
INITIAL_CAPITAL = 100000
FEES = 0.001

trades = run_backtest(df, entries, INITIAL_CAPITAL, FEES)

if len(trades) > 0:
    print("TRADE LOG - 2025")
    print("="*110)
    print(f"{'Entry':<12} {'Exit':<12} {'Entry $':>10} {'Exit $':>10} {'Gross':>8} {'Net':>8} {'Days':>6} {'Exit Reason':<12}")
    print("-"*110)
    
    for _, t in trades.iterrows():
        print(f"{str(t['entry_date'].date()):<12} {str(t['exit_date'].date()):<12} "
              f"{t['entry_price']:>10,.0f} {t['exit_price']:>10,.0f} "
              f"{t['gross_return']*100:>+7.0f}% {t['net_return']*100:>+7.0f}% "
              f"{t['days_held']:>6} {t['exit_reason']:<12}")
else:
    print("No trades in 2025!")

In [ ]:
# Performance metrics
if len(trades) > 0:
    total_return = (trades['equity_after'].iloc[-1] / INITIAL_CAPITAL) - 1
    win_rate = (trades['net_return'] > 0).mean()
    
    # Buy & hold comparison
    bh_return = (df['price'].iloc[-1] / df['price'].iloc[0]) - 1
    
    print("\n" + "="*60)
    print("2025 PERFORMANCE SUMMARY")
    print("="*60)
    
    print(f"\n📊 STRATEGY")
    print(f"   Total Trades: {len(trades)}")
    print(f"   Win Rate: {win_rate*100:.0f}%")
    print(f"   Total Return: {total_return*100:+.1f}%")
    print(f"   Final Equity: ${trades['equity_after'].iloc[-1]:,.0f}")
    
    print(f"\n📈 BUY & HOLD")
    print(f"   BTC Jan 1: ${df['price'].iloc[0]:,.0f}")
    print(f"   BTC Now: ${df['price'].iloc[-1]:,.0f}")
    print(f"   Return: {bh_return*100:+.1f}%")
    
    print(f"\n🎯 COMPARISON")
    print(f"   Strategy: {total_return*100:+.1f}%")
    print(f"   Buy & Hold: {bh_return*100:+.1f}%")
    print(f"   Difference: {(total_return - bh_return)*100:+.1f}%")
    
    if total_return > bh_return:
        print(f"\n   ✅ Strategy BEAT Buy & Hold!")
    else:
        print(f"\n   ❌ Strategy UNDERPERFORMED Buy & Hold")
else:
    bh_return = (df['price'].iloc[-1] / df['price'].iloc[0]) - 1
    print(f"\nNo trades in 2025")
    print(f"BTC Buy & Hold: {bh_return*100:+.1f}%")

In [ ]:
# Visualization
if len(trades) > 0:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.6, 0.4],
                        subplot_titles=['BTC Price with Trades', 'MVRV'])
    
    # Price
    fig.add_trace(go.Scatter(
        x=df.index, y=df['price'],
        name='BTC Price', line=dict(color='orange', width=1)
    ), row=1, col=1)
    
    # Entry markers
    fig.add_trace(go.Scatter(
        x=trades['entry_date'], y=trades['entry_price'],
        mode='markers', name='Entry',
        marker=dict(color='green', size=12, symbol='triangle-up')
    ), row=1, col=1)
    
    # Exit markers
    colors = {'mvrv_trail': 'blue', 'stop_loss': 'red', 'end_of_data': 'gray'}
    for reason in trades['exit_reason'].unique():
        mask = trades['exit_reason'] == reason
        fig.add_trace(go.Scatter(
            x=trades.loc[mask, 'exit_date'], y=trades.loc[mask, 'exit_price'],
            mode='markers', name=f'Exit ({reason})',
            marker=dict(color=colors.get(reason, 'gray'), size=12, symbol='triangle-down')
        ), row=1, col=1)
    
    # MVRV
    fig.add_trace(go.Scatter(
        x=df.index, y=df['mvrv'],
        name='MVRV', line=dict(color='purple', width=1)
    ), row=2, col=1)
    fig.add_hline(y=2.0, line_dash='dash', line_color='red', row=2, col=1,
                  annotation_text='MVRV Trigger (2.0)')
    
    fig.update_layout(height=700, title_text='STRAT-002 Performance in 2025')
    fig.show()

In [ ]:
# Current status
print("\n" + "="*60)
print("CURRENT STATUS")
print("="*60)

if len(trades) > 0:
    latest_trade = trades.iloc[-1]
    
    if latest_trade['exit_reason'] == 'end_of_data':
        print(f"\n🔵 CURRENTLY IN TRADE")
        print(f"   Entry: {latest_trade['entry_date'].date()} at ${latest_trade['entry_price']:,.0f}")
        print(f"   Current Price: ${df['price'].iloc[-1]:,.0f}")
        current_pnl = (df['price'].iloc[-1] / latest_trade['entry_price']) - 1
        print(f"   Unrealized P&L: {current_pnl*100:+.1f}%")
        print(f"   Current MVRV: {df['mvrv'].iloc[-1]:.2f}")
        
        if df['mvrv'].iloc[-1] >= 2.0:
            print(f"   ⚠️ MVRV > 2.0 - Trailing stop ACTIVE")
        else:
            print(f"   Waiting for MVRV > 2.0 to activate trail")
    else:
        print(f"\n⚪ NOT IN TRADE")
        print(f"   Last exit: {latest_trade['exit_date'].date()} ({latest_trade['exit_reason']})")

# Check current signals
print(f"\n📡 CURRENT SIGNALS")
print(f"   SOPR: {df['sopr'].iloc[-1]:.4f} {'✅ < 1' if df['sopr'].iloc[-1] < 1 else '❌ >= 1'}")
print(f"   STH-SOPR: {df['sopr_sth'].iloc[-1]:.4f} {'✅ < 1' if df['sopr_sth'].iloc[-1] < 1 else '❌ >= 1'}")
print(f"   RL Z-Score: {df['rl_zscore'].iloc[-1]:.2f} {'✅ > 0.5' if df['rl_zscore'].iloc[-1] > 0.5 else '❌ <= 0.5'}")

if df['sopr'].iloc[-1] < 1 and df['sopr_sth'].iloc[-1] < 1 and df['rl_zscore'].iloc[-1] > 0.5:
    print(f"\n   🚨 ENTRY SIGNAL ACTIVE!")
else:
    print(f"\n   No entry signal currently")